# RAG Pipeline - Document Ingestion
This notebook handles the first stage of the RAG (Retrieval-Augmented Generation) pipeline: **loading documents and creating a vector store**.

## Workflow
1. **Authentication** - Connect to HuggingFace Hub
2. **Setup** - Install dependencies and configure environment
3. **Ingestion** - Load documents, chunk them, and create embeddings

In [1]:
from dotenv import load_dotenv
import os

load_dotenv()   # reads .env from the current working directory
hf_token = os.getenv("HF_TOKEN")

from huggingface_hub import login
login(token=hf_token)

/Users/adel/Desktop/Rag-project/rag-env/lib/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Note: Environment variable`HF_TOKEN` is set and is the current active token independently from the token you've just configured.


In [2]:
!python -m ipykernel install --user --name=rag-env --display-name "RAG Project (rag-env)"

Installed kernelspec rag-env in /Users/adel/Library/Jupyter/kernels/rag-env


## Setup & Dependencies

In [3]:
import sys
!{sys.executable} -m pip install torch torchvision torchaudio
!{sys.executable} -m pip install transformers accelerate
!{sys.executable} -m pip install sentence-transformers
!{sys.executable} -m pip install faiss-cpu chromadb
!{sys.executable} -m pip install langchain langchain-community langchain-huggingface langchain-text-splitters
!{sys.executable} -m pip install pypdf unstructured python-docx
!{sys.executable} -m pip install huggingface_hub datasets
!{sys.executable} -m pip install einops scipy tiktoken python-dotenv
!{sys.executable} -m pip install llama-cpp-python


[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: pip install --upgrade pip

[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: pip install --upgrade pip

[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: pip install --upgrade pip

[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: pip install --upgrade pip

[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: pip install --upgrade pip

[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: pip install --upgrade pip

[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: pip install --upgrade pip

[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: pip install --upgrade pip

[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: pip install --upgrade pip


In [4]:
import torch
import platform

print(f"Python:       {platform.python_version()}")
print(f"PyTorch:      {torch.__version__}")
print(f"MPS available: {torch.backends.mps.is_available()}")
print(f"MPS built:     {torch.backends.mps.is_built()}")

device = "mps" if torch.backends.mps.is_available() else "cpu"
print(f"\n Using device: {device}")

Python:       3.14.3
PyTorch:      2.11.0
MPS available: True
MPS built:     True

 Using device: mps


In [5]:
!pwd

/Users/adel/Desktop/Rag-project/notebooks


In [6]:
import os
os.chdir(os.path.dirname(os.path.abspath("__file__")))  # sets cwd to notebook location
print(os.getcwd()) 

/Users/adel/Desktop/Rag-project/notebooks


## Document Ingestion Pipeline

In [7]:
from pathlib import Path
from langchain_community.document_loaders import (
    PyPDFLoader,
    TextLoader,
    Docx2txtLoader,
    UnstructuredFileLoader,
)
from langchain_text_splitters import RecursiveCharacterTextSplitter

# Prefer the new package when available (LangChain deprecation path)
try:
    from langchain_chroma import Chroma
except Exception:
    from langchain_community.vectorstores import Chroma

from langchain_huggingface import HuggingFaceEmbeddings
import os

# Define paths
DATA_DIR = Path("../data/documents/")
VECTORSTORE_DIR = Path("../data/vectorstore")

# Create output directory if it doesn't exist
VECTORSTORE_DIR.mkdir(parents=True, exist_ok=True)

# Document loader mapping
LOADERS = {
    '.pdf': PyPDFLoader,
    '.txt': TextLoader,
    '.docx': Docx2txtLoader,
    '.doc': Docx2txtLoader,
}

/Users/adel/Desktop/Rag-project/rag-env/lib/python3.14/site-packages/langchain_core/_api/deprecation.py:25: UserWarning: Core Pydantic V1 functionality isn't compatible with Python 3.14 or greater.
  from pydantic.v1.fields import FieldInfo as FieldInfoV1


In [8]:
def load_documents(directory):
    """Load all supported documents from directory"""
    documents = []
    
    if not directory.exists():
        print(f"⚠️  Directory not found: {directory}")
        return documents
    
    # Get all files in directory
    for file_path in directory.glob('*'):
        if file_path.is_file():
            suffix = file_path.suffix.lower()
            
            if suffix in LOADERS:
                try:
                    loader_class = LOADERS[suffix]
                    loader = loader_class(str(file_path))
                    docs = loader.load()
                    documents.extend(docs)
                    print(f"✅ Loaded: {file_path.name} ({len(docs)} pages/sections)")
                except Exception as e:
                    print(f"❌ Error loading {file_path.name}: {str(e)}")
            else:
                print(f"⏭️  Skipped: {file_path.name} (unsupported format)")
    
    return documents


def chunk_documents(documents, chunk_size=1000, chunk_overlap=200):
    """Split documents into chunks"""
    text_splitter = RecursiveCharacterTextSplitter(
        chunk_size=chunk_size,
        chunk_overlap=chunk_overlap,
        separators=["\n\n", "\n", " ", ""]
    )
    chunks = text_splitter.split_documents(documents)
    return chunks


def create_vectorstore(chunks, embedding_model, persist_dir, device: str = "cpu", reset: bool = False):
    """Create and persist a Chroma vector store.

    Notes:
    - The embedding model used here MUST match the one used when re-loading the store.
    - If you re-run ingestion into the same folder, Chroma can accumulate duplicates.
      Set reset=True to rebuild cleanly.
    """
    if reset and persist_dir.exists():
        import shutil
        shutil.rmtree(persist_dir)
        persist_dir.mkdir(parents=True, exist_ok=True)

    embeddings = HuggingFaceEmbeddings(
        model_name=embedding_model,
        model_kwargs={"device": device},
    )

    vectorstore = Chroma.from_documents(
        documents=chunks,
        embedding=embeddings,
        persist_directory=str(persist_dir),
        collection_name="rag",
    )

    return vectorstore

In [ ]:
print(f"📂 Looking for documents in: {DATA_DIR.absolute()}")
print(f"📊 Documents found: {list(DATA_DIR.glob('*'))}\n")

# Load documents
raw_documents = load_documents(DATA_DIR)
print(f"\n📄 Total documents loaded: {len(raw_documents)}")

if raw_documents:
    # Split documents into chunks
    chunks = chunk_documents(raw_documents, chunk_size=1000, chunk_overlap=200)
    print(f"✂️  Split into {len(chunks)} chunks")
    
    # Create vector store
    print("\n🔄 Creating vector store...")
    vectorstore = create_vectorstore(
        chunks=chunks,
        embedding_model="sentence-transformers/all-MiniLM-L6-v2",
        persist_dir=VECTORSTORE_DIR,
        device=device,
        reset=True,  # set to False if you want to append
    )
    
    print(f"✅ Vector store created with {len(chunks)} vectors!")
    
    # Test retrieval
    query = "What is this document about?"
    results = vectorstore.similarity_search(query, k=3)
    print(f"\n🔍 Test query: '{query}'")
    print(f"Found {len(results)} relevant chunks")
else:
    print("\n⚠️  No documents found. Add PDF, TXT, or DOCX files to data/documents/ folder")

## Summary
✅ **Ingestion Complete**

The vector store has been created and persisted to `data/vectorstore/`. You can now use it in the next stage of the pipeline:
- **Next Step**: Run `02_embed.ipynb` to generate additional embeddings if needed
- **Vector Store Location**: `data/vectorstore/` (ChromaDB format)
- **Documents Processed**: Check the output above for details